# B v8 — designated nighttime dataset preflight

Continuation of the B v8 reviewed prototype. Inspection only by default: no detector, tracker, ReID, enhancement inference or counting. Historic B v7/B v8 baselines are not modified. Review-Filter exports are validated separately; no eight-video assumption.

In [ ]:
from pathlib import Path
import json, sys
DATASET_ROOT = None  # Discover actual mount below; do not assume a path.
OUTPUT_ROOT = Path("/kaggle/working/nighttime_input_inspection")
RUN_INFERENCE = False
FILTER_CONFIG_PATH = None  # Supply exported review-filter JSON only after review.

In [ ]:
INPUT_ROOT=Path("/kaggle/input")
assert INPUT_ROOT.is_dir(), "Kaggle input is not mounted"
print("Actual top-level mounts:", sorted(p.name for p in INPUT_ROOT.iterdir()))
candidates = [p for p in INPUT_ROOT.rglob("video-traffic-malam") if p.is_dir()]
if DATASET_ROOT is None:
    assert len(candidates)==1, f"Set DATASET_ROOT from actual mounts; found {candidates}"
    DATASET_ROOT=candidates[0]
assert DATASET_ROOT.is_dir()
print("Actual dataset:", DATASET_ROOT)
print("Actual files:", [(str(p.relative_to(DATASET_ROOT)),p.stat().st_size) for p in sorted(DATASET_ROOT.rglob("*")) if p.is_file()])

In [ ]:
HELPER_ROOT=Path("/kaggle/working/nighttime_preflight_helpers")
HELPER_ROOT.mkdir(exist_ok=True)
(HELPER_ROOT/"nighttime_input_inspection.py").write_text('"""Read-only source inspection. No model, enhancement, tracking or counting."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport time\n\nVIDEO_EXTENSIONS = {\'.mp4\', \'.avi\', \'.mov\', \'.mkv\', \'.webm\', \'.m4v\', \'.mts\', \'.mpeg\', \'.mpg\'}\nDATASET = \'chrisbiran/video-traffic-malam\'\n\n\ndef file_sha256(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef positive(value):\n    return float(value) if isinstance(value, (int, float)) and math.isfinite(value) and value > 0 else None\n\n\ndef probe(path):\n    """Keep absent metadata unknown; ffprobe durations are not physical ground truth."""\n    if not shutil.which(\'ffprobe\'):\n        return {\'available\': False, \'error\': \'ffprobe not installed\', \'stream\': {}, \'format\': {}}\n    command = [\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'v:0\', \'-show_streams\',\n               \'-show_format\', \'-of\', \'json\', str(path)]\n    result = subprocess.run(command, capture_output=True, text=True, timeout=60)\n    if result.returncode:\n        return {\'available\': True, \'error\': result.stderr.strip(), \'stream\': {}, \'format\': {}}\n    data = json.loads(result.stdout)\n    return {\'available\': True, \'error\': None, \'stream\': next(iter(data.get(\'streams\', [])), {}),\n            \'format\': data.get(\'format\', {})}\n\n\ndef fraction_rate(text):\n    try:\n        a, b = text.split(\'/\')\n        return positive(float(a) / float(b))\n    except (AttributeError, TypeError, ValueError, ZeroDivisionError):\n        return None\n\n\ndef inspect_video(path, output, dataset_root):\n    import cv2\n    import numpy as np\n    from PIL import Image, ImageDraw\n\n    started = time.perf_counter()\n    path, output = Path(path), Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    before = file_sha256(path)\n    info = probe(path)\n    stream = info[\'stream\']\n    cap = cv2.VideoCapture(str(path))\n    opened = cap.isOpened()\n    declared = positive(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else None\n    declared_count = int(declared) if declared is not None and declared.is_integer() else None\n    fps_opencv = positive(cap.get(cv2.CAP_PROP_FPS)) if opened else None\n    target_count = declared_count\n    targets = sorted(set([0, *[round((target_count - 1) * p) for p in [.25, .5, .75, 1]]])) if target_count else [0]\n    samples, decoded, dimensions, first_failure = [], 0, set(), None\n    last_frame = None\n    while opened:\n        ok, frame = cap.read()\n        if not ok:\n            first_failure = decoded\n            break\n        last_frame = frame\n        height, width = frame.shape[:2]\n        dimensions.add((width, height))\n        if decoded in targets:\n            name = f\'frame_{decoded:06d}.png\'\n            assert cv2.imwrite(str(output / name), frame), \'Could not save native frame\'\n            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n            samples.append({\'frame_index\': decoded, \'native_image\': name, \'image_sha256\': file_sha256(output/name),\n                            \'decoded_bgr_sha256\': hashlib.sha256(frame.tobytes()).hexdigest(),\n                            \'reported_position_msec\': cap.get(cv2.CAP_PROP_POS_MSEC),\n                            \'luma_p05_p50_p95\': [float(x) for x in np.percentile(gray, [5, 50, 95])],\n                            \'fraction_luma_le_16\': float(np.mean(gray <= 16)),\n                            \'fraction_luma_ge_240\': float(np.mean(gray >= 240)),\n                            \'metric_limits\': \'Pixel summaries only; no detection/noise/visibility quality threshold.\'})\n        decoded += 1\n    cap.release()\n    # Retain actual end even if header count was inaccurate; do not fabricate 25/50/75% coverage.\n    if decoded and decoded-1 not in [s[\'frame_index\'] for s in samples]:\n        name = f\'frame_{decoded-1:06d}.png\'\n        assert cv2.imwrite(str(output/name), last_frame)\n        samples.append({\'frame_index\': decoded-1, \'native_image\': name, \'image_sha256\': file_sha256(output/name),\n                        \'decoded_bgr_sha256\': hashlib.sha256(last_frame.tobytes()).hexdigest(),\n                        \'note\': \'Actual last decodable frame; header-based sampling may be incomplete.\'})\n    if samples:\n        sheet = Image.new(\'RGB\', (360 * len(samples), 292), \'#202020\')\n        draw = ImageDraw.Draw(sheet)\n        for i, sample in enumerate(samples):\n            im = Image.open(output/sample[\'native_image\']).convert(\'RGB\')\n            im.thumbnail((360, 250))\n            sheet.paste(im, (i*360, 24))\n            draw.text((i*360+5, 5), f"source frame {sample[\'frame_index\']}", fill=\'white\')\n        sheet.save(output/\'contact_sheet.jpg\', quality=92)\n    after = file_sha256(path)\n    assert before == after, \'Source video changed during read-only inspection\'\n    if not opened or decoded == 0:\n        status = \'UNREADABLE_OR_NO_DECODABLE_FRAMES\'\n    elif declared_count is not None and decoded != declared_count:\n        status = \'DECODE_HEADER_COUNT_MISMATCH_REVIEW_REQUIRED\'\n    else:\n        status = \'SEQUENTIAL_DECODE_TO_EOF\'\n    duration = None\n    for value in [stream.get(\'duration\'), info[\'format\'].get(\'duration\')]:\n        try:\n            duration = positive(float(value))\n        except (TypeError, ValueError):\n            pass\n        if duration is not None:\n            break\n    result = {\'filename\': path.name, \'relative_path\': str(path.relative_to(dataset_root)),\n              \'extension\': path.suffix, \'size_bytes\': path.stat().st_size, \'source_sha256\': before,\n              \'source_unchanged\': True, \'width\': stream.get(\'width\'), \'height\': stream.get(\'height\'),\n              \'fps_reported_ffprobe\': fraction_rate(stream.get(\'avg_frame_rate\')),\n              \'fps_reported_opencv\': fps_opencv, \'duration_reported_seconds\': duration,\n              \'duration_from_decoded_count_and_reported_fps_seconds\': decoded / fps_opencv if fps_opencv else None,\n              \'codec\': stream.get(\'codec_name\'), \'codec_tag\': stream.get(\'codec_tag_string\'),\n              \'pixel_format\': stream.get(\'pix_fmt\'), \'frame_count_reported_opencv\': declared_count,\n              \'frame_count_decoded\': decoded, \'decoded_dimensions\': sorted(dimensions),\n              \'decode_status\': status, \'first_unsuccessful_read_frame\': first_failure,\n              \'decode_limit\': \'OpenCV sequential EOF plus header comparison; not a bitstream corruption certification.\',\n              \'ffprobe\': info, \'samples\': samples, \'sampling\': \'Sequential source indices at header-based 0/25/50/75/100%; actual end retained.\',\n              \'condition\': None, \'condition_assignment\': \'UNASSIGNED_REQUIRES_REVIEW_NOT_FILENAME_INFERENCE\',\n              \'wall_seconds\': time.perf_counter()-started}\n    (output/\'inspection.json\').write_text(json.dumps(result, indent=2, allow_nan=False)+\'\\n\')\n    return result\n\n\ndef inspect_dataset(dataset_root, output, *, execution_context):\n    root, output = Path(dataset_root).resolve(), Path(output).resolve()\n    if not root.is_dir():\n        raise FileNotFoundError(f\'Actual dataset directory unavailable: {root}\')\n    if root == output or root in output.parents:\n        raise ValueError(\'Inspection output must not modify the original dataset tree\')\n    files = sorted(p for p in root.rglob(\'*\') if p.is_file())\n    if not files:\n        raise ValueError(\'Dataset directory contains no files\')\n    output.mkdir(parents=True, exist_ok=False)\n    inventory, unsupported = [], []\n    for i, path in enumerate(files):\n        if path.suffix.lower() not in VIDEO_EXTENSIONS:\n            unsupported.append({\'relative_path\': str(path.relative_to(root)), \'size_bytes\': path.stat().st_size,\n                                \'reason\': \'Extension not in declared video list; not silently treated as video\'})\n            continue\n        target = output/f\'video_{i:03d}\'\n        row = inspect_video(path, target, root)\n        row[\'evidence_directory\'] = target.name\n        inventory.append(row)\n        print(json.dumps({\'filename\': path.name, \'decoded_frames\': row[\'frame_count_decoded\'], \'status\': row[\'decode_status\']}), flush=True)\n    import cv2\n    environment = {\'python\': platform.python_version(), \'opencv\': cv2.__version__, \'execution_context\': execution_context,\n                   \'actual_dataset_directory\': str(root), \'kaggle_input_exists\': Path(\'/kaggle/input\').is_dir(),\n                   \'cpu_count\': __import__(\'os\').cpu_count(),\n                   \'kaggle_input_top_level_actual\': sorted(p.name for p in Path(\'/kaggle/input\').iterdir()) if Path(\'/kaggle/input\').is_dir() else [],\n                   \'gpu_required_for_inspection\': False}\n    report = {\'schema_version\': \'official_night_input_inspection_v1\', \'designated_dataset\': DATASET,\n              \'environment\': environment, \'actual_files\': [{\'path\': str(p.relative_to(root)), \'size_bytes\': p.stat().st_size} for p in files],\n              \'video_count\': len(inventory), \'inventory\': inventory, \'unsupported_files\': unsupported,\n              \'unreadable_or_mismatched\': [r[\'filename\'] for r in inventory if r[\'decode_status\'] != \'SEQUENTIAL_DECODE_TO_EOF\'],\n              \'manual_visual_review_status\': \'PENDING\', \'new_model_execution\': False,\n              \'review_filter_implemented_or_validated_by_this_inspection\': False,\n              \'counting_smoke_test_executed\': False, \'production_reliability_established\': False}\n    (output/\'inventory.json\').write_text(json.dumps(report, indent=2, allow_nan=False)+\'\\n\')\n    import csv\n    fields = [\'filename\', \'extension\', \'size_bytes\', \'duration_reported_seconds\', \'fps_reported_ffprobe\', \'width\', \'height\',\n              \'codec\', \'frame_count_decoded\', \'decode_status\', \'source_sha256\', \'evidence_directory\']\n    with (output/\'inventory.csv\').open(\'w\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction=\'ignore\'); writer.writeheader(); writer.writerows(inventory)\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--dataset-root\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--execution-context\', required=True, choices=[\'codex_downloaded_dataset\', \'actual_kaggle_mount\'])\n    args = parser.parse_args()\n    inspect_dataset(args.dataset_root, args.output, execution_context=args.execution_context)\n')
(HELPER_ROOT/"nighttime_review_filter.py").write_text('"""Explicit photometric contract shared with results/nighttime_review_filter.js.\n\nNew experiment, never an automatic replacement for the retained raw baseline.\n"""\nimport copy\nimport hashlib\nimport json\nimport math\n\nSCHEMA = \'night_review_filter_v1\'\nORDER = [\'gamma\', \'exposure\', \'brightness\', \'shadows\', \'sharpness\']\nCONDITIONS = [f\'{day}_{period}\' for day in [\'weekday\', \'weekend\'] for period in [\'pagi\', \'siang\', \'sore\', \'malam\']]\nRAW = {\'gamma\': 1.0, \'exposure\': 0.0, \'sharpness\': 0.0, \'brightness\': 0.0, \'shadows\': 0.0}\nRANGES = {\'gamma\': (.2, 3.), \'exposure\': (-3., 3.), \'sharpness\': (0., 2.), \'brightness\': (-100., 100.), \'shadows\': (-100., 100.)}\n\n\ndef validate_settings(settings):\n    if not isinstance(settings, dict) or set(settings) != set(RAW):\n        raise ValueError(\'Exactly gamma/exposure/sharpness/brightness/shadows required\')\n    for key, (low, high) in RANGES.items():\n        value = settings[key]\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not low <= value <= high:\n            raise ValueError(f\'Unsupported {key}: expected finite number in [{low},{high}]\')\n    return copy.deepcopy(settings)\n\n\ndef validate_config(config):\n    if config.get(\'schema_version\') != SCHEMA or config.get(\'operation_order\') != ORDER:\n        raise ValueError(\'Unknown schema or transform order\')\n    if set(config.get(\'profiles\', {})) != set(CONDITIONS):\n        raise ValueError(\'Eight independent condition profiles required, not eight assumed input videos\')\n    for settings in config[\'profiles\'].values():\n        validate_settings(settings)\n    if not isinstance(config.get(\'videos\'), list):\n        raise ValueError(\'Explicit video list required\')\n    hashes = set()\n    import re\n    for video in config[\'videos\']:\n        sha = video.get(\'source_sha256\')\n        if not isinstance(sha, str) or not re.fullmatch(\'[0-9a-f]{64}\', sha) or sha in hashes:\n            raise ValueError(\'Unique actual video SHA256 required\')\n        hashes.add(sha)\n        if not isinstance(video.get(\'filename\'), str) or not video[\'filename\']:\n            raise ValueError(\'Actual video filename required\')\n        if video.get(\'condition\') is not None and video[\'condition\'] not in CONDITIONS:\n            raise ValueError(\'Unknown condition; leave null instead of guessing\')\n        validate_settings(video[\'settings\'])\n    return copy.deepcopy(config)\n\n\ndef settings_for_video(config, source_sha256):\n    validated = validate_config(config)\n    for video in validated[\'videos\']:\n        if video[\'source_sha256\'] == source_sha256:\n            return copy.deepcopy(video[\'settings\'])\n    raise ValueError(\'Source video has no explicit saved settings; do not silently use another slot\')\n\n\ndef apply_filter(frame_bgr, settings):\n    """Preserves BGR order and native dimensions; half-up rounding after each stage."""\n    import numpy as np\n    s = validate_settings(settings)\n    if not isinstance(frame_bgr, np.ndarray) or frame_bgr.dtype != np.uint8 or frame_bgr.ndim != 3 or frame_bgr.shape[2] != 3 or min(frame_bgr.shape[:2]) < 1:\n        raise ValueError(\'Nonempty HxWx3 BGR uint8 required\')\n    def quantize(x):\n        return np.floor(np.clip(x, 0, 255) + .5).astype(np.uint8)\n    result = frame_bgr.copy()\n    if s[\'gamma\'] != 1:\n        result = quantize(255. * (result.astype(np.float64)/255.) ** s[\'gamma\'])\n    if s[\'exposure\']:\n        result = quantize(result.astype(np.float64) * 2. ** s[\'exposure\'])\n    if s[\'brightness\']:\n        result = quantize(result.astype(np.float64) + s[\'brightness\'])\n    if s[\'shadows\']:\n        x = result.astype(np.float64)\n        result = quantize(x + .64 * s[\'shadows\'] * (1. - x/255.) ** 2)\n    if s[\'sharpness\']:\n        x = result.astype(np.float64)\n        padded = np.pad(x, ((1, 1), (1, 1), (0, 0)), mode=\'edge\')\n        smooth = np.zeros_like(x)\n        for j, wy in enumerate([1, 2, 1]):\n            for i, wx in enumerate([1, 2, 1]):\n                smooth += padded[j:j+x.shape[0], i:i+x.shape[1]] * wy * wx / 16.\n        result = quantize(x + s[\'sharpness\'] * (x - smooth))\n    return result\n\n\ndef prepare_inference_frame(raw_frame_bgr, config, source_sha256):\n    """Use this SAME result for detector and ReID frame input, never raw in one branch."""\n    result = apply_filter(raw_frame_bgr, settings_for_video(config, source_sha256))\n    return result, {\'schema_version\': SCHEMA, \'source_sha256\': source_sha256,\n                    \'operation_order\': ORDER, \'settings\': settings_for_video(config, source_sha256),\n                    \'output_bgr_sha256\': hashlib.sha256(result.tobytes()).hexdigest(),\n                    \'config_sha256\': hashlib.sha256(json.dumps(config, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()}\n')
sys.path.insert(0,str(HELPER_ROOT))
import nighttime_input_inspection as inspection
import nighttime_review_filter as review_filter

In [ ]:
report=inspection.inspect_dataset(DATASET_ROOT,OUTPUT_ROOT,execution_context="actual_kaggle_mount")
print({"videos":report["video_count"],"decode_review_required":report["unreadable_or_mismatched"],"output":str(OUTPUT_ROOT),"inference_executed":False})

In [ ]:
if FILTER_CONFIG_PATH is not None:
    saved=review_filter.validate_config(json.loads(Path(FILTER_CONFIG_PATH).read_text()))
    actual_hashes={v["source_sha256"] for v in report["inventory"]}
    assert all(v["source_sha256"] in actual_hashes for v in saved["videos"]), "Saved filter source differs from inspected dataset"
    print("Filter configuration schema and input identity checked; no production validation implied")
else:
    print("SMOKE BLOCKED: saved Review-Filter settings and reviewed per-video geometry required before model loading")
assert not RUN_INFERENCE, "This inspection notebook intentionally has no inference entrypoint. Freeze source/filter/geometry first."

Review native frames and `inventory.json` first. Use `results/review-filter.html` from GitHub/workspace to export per-video filter settings; compare with raw on exactly the same continuous source scope. A full counting smoke must use a reviewed native geometry and preserve B v7/B v8 safeguards. Passing decode or filter parity alone does not establish nighttime counting reliability. Area Coverage and PPAC are outside this notebook.